**Business problem: A telecom company is losing customers. Which customers are likely to leave, and what factors drive them to leave?**

**Goal: Find the main reasons for churn and recommend ways to reduce it.**

In [23]:
import pandas as pd
import numpy as np

In [24]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [25]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [26]:
df.shape

(7043, 21)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [28]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## Dataset Overview

**Size of the dataset**
- The dataset has **7,043 rows** (each row is one customer) and **21 columns**.
- Of these, 1 column is an ID (customerID), 1 is the target (Churn), and the remaining **19 are features** that describe the customer.

**Target column**
- The target is **Churn**, with two values: **Yes** (the customer left) and **No** (the customer stayed).
- This is a **binary classification** problem: we want to predict Yes or No for each customer.

**Numerical columns**
- tenure: number of months the customer has stayed with the company
- MonthlyCharges: the amount charged to the customer each month
- TotalCharges: the total amount charged over the customer's lifetime (stored as text at first, converted to numbers during cleaning)

**Categorical columns**
- Customer details: gender, SeniorCitizen (0/1, treated as Yes/No), Partner, Dependents
- Services: PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies
- Account details: Contract, PaperlessBilling, PaymentMethod
- Target: Churn

**Identifier column**
- customerID is a unique ID for each customer. It has no predictive value, so it will be dropped.

In [29]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [30]:
df.duplicated().sum()

np.int64(0)

In [31]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


In [32]:
df['Churn'].value_counts(normalize=True) * 100

,proportion
Churn,
No,73.463013
Yes,26.536987


In [33]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [34]:
df = df.dropna(subset=['TotalCharges'])

   **Missing values:** TotalCharges had 11 blanks. These are customers with tenure = 0 (brand new, not yet billed). Since this is only 11 of 7,043 rows (0.16%), I dropped them. The cleaned dataset has 7,032 rows.

In [35]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [36]:
df = df.drop('customerID', axis=1)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())

gender ['Female' 'Male']
SeniorCitizen ['No' 'Yes']
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
PhoneService ['No' 'Yes']
MultipleLines ['No phone service' 'No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes' 'No internet service']
OnlineBackup ['Yes' 'No' 'No internet service']
DeviceProtection ['No' 'Yes' 'No internet service']
TechSupport ['No' 'Yes' 'No internet service']
StreamingTV ['No' 'Yes' 'No internet service']
StreamingMovies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn ['No' 'Yes']


In [37]:
df.to_csv("churn_cleaned.csv", index=False)

In [38]:
print(df.groupby('Contract')['Churn'].value_counts(normalize=True))

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887228
                Yes      0.112772
Two year        No       0.971513
                Yes      0.028487
Name: proportion, dtype: float64


In [39]:
print(df.groupby('InternetService')['Churn'].value_counts(normalize=True))

InternetService  Churn
DSL              No       0.810017
                 Yes      0.189983
Fiber optic      No       0.581072
                 Yes      0.418928
No               No       0.925658
                 Yes      0.074342
Name: proportion, dtype: float64


In [40]:
df.groupby('InternetService')['MonthlyCharges'].mean()

,MonthlyCharges
InternetService,
DSL,58.088017
Fiber optic,91.500129
No,21.076283


- Fiber optic customers also pay much more per month (about $91.50 vs $58.09 for DSL), so higher prices may be one reason for their higher churn. This is an association, not proof, and I will test it further in the next stage.

In [41]:
df.groupby('Churn')[['tenure', 'MonthlyCharges']].mean()

,tenure,MonthlyCharges
Churn,,
No,37.650010,61.307408
Yes,17.979133,74.441332


## First Findings

**1. Overall churn**
- 1,869 of 7,043 customers churned, an overall churn rate of about **26.5%**. Roughly 1 in 4 customers leaves.
- The data is imbalanced (73.5% stayed vs 26.5% left), which I need to keep in mind when building the model.

**2. Churn by contract type**
- Month-to-month customers churn at **42.7%**, one-year at **11.3%**, and two-year at only **2.8%**.
- Month-to-month customers are about 15 times more likely to leave than two-year customers (42.7% vs 2.8%). Without a long commitment, customers can leave easily, so contract length is one of the strongest churn signals so far.

**3. Churn by internet service**
- Fiber optic customers churn at **41.9%**, DSL at **19.0%**, and customers with no internet service at only **7.4%**.
- Fiber optic customers are more than twice as likely to leave as DSL customers (41.9% vs 19.0%).
- Fiber optic is the premium service, so this could point to higher prices or service quality issues. The data doesn't show the reason yet, so this needs a closer look in Week 2.

**4. Tenure**
- Customers who left had an average tenure of about **18 months**, compared with about **38 months** for those who stayed, so churned customers had roughly half the tenure.
- Most churn happens early in the customer relationship, so the first year or two is the critical window for retention.

**5. Monthly charges**
- Customers who left pay about **$74.44** per month on average, compared with **$61.31** for those who stayed. That is about **$13 (21%) more** per month.
- Higher bills are linked with a higher chance of leaving.

**Takeaway:** Churn is concentrated among newer customers with higher monthly bills. The next step is to confirm which contract and service types drive this.